In [1]:
import os
import time
import uuid
from statistics import mean

# Comparing read/write speeds for Google Filestore versus a FUSE storage bucket

Andrea has added a Google storage bucket with `readwritemany` permissions to the JupyterHub using the FUSE CSI driver. Kim tried this previously but was prevented due to issues with the CSI driver, which have now been fixed. See the PR [here](https://github.com/NIVANorge/niva_jupyter_hub/pull/45#event-25960853414) for details.

The FUSE storage bucket is much cheaper than Filestore and will be a good option for long-term storage of large files. This notebook compares the read and write performance of the Filestore with the new FUSE storage bucket.

**Most of the code and interpretation in this notebook is adapted from Copilot - use with cautrion!**

In [2]:
# Paths
filestore_path = r"/home/jovyan/shared/common/JES/test"
fuse_path = r"/home/jovyan/shared/disk/JES/test"

# Config
FILE_SIZE_MB = 2048  # 2 GB (adjust if needed)
BLOCK_SIZES = [
    4 * 1024,  # 4 KB
    64 * 1024,  # 64 KB
    1 * 1024 * 1024,  # 1 MB
    4 * 1024 * 1024,  # 4 MB
]
REPEATS = 3

In [3]:
def benchmark_write(path, block_size, file_size_mb):
    filename = f"test_{uuid.uuid4().hex}.bin"
    filepath = os.path.join(path, filename)

    data = os.urandom(block_size)
    total_bytes = file_size_mb * 1024 * 1024
    written = 0

    start = time.time()

    with open(filepath, "wb") as f:
        while written < total_bytes:
            f.write(data)
            written += len(data)
        f.flush()
        os.fsync(f.fileno())

    elapsed = time.time() - start
    speed = (total_bytes / (1024**2)) / elapsed  # MB/s
    return speed, filepath


def benchmark_read(filepath, block_size):
    start = time.time()
    total = 0

    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            total += len(chunk)

    elapsed = time.time() - start
    speed = (total / (1024**2)) / elapsed  # MB/s
    return speed


def cleanup(filepath):
    try:
        os.remove(filepath)
    except Exception as e:
        print("Cleanup error:", e)


def metadata_test(path, n_files=2000):
    print(f"\nMetadata test ({n_files} files): {path}")
    tmp_files = []

    start = time.time()

    # Create files
    for i in range(n_files):
        fname = os.path.join(path, f"tmp_{uuid.uuid4().hex}.txt")
        with open(fname, "w") as f:
            f.write("x")
        tmp_files.append(fname)

    create_time = time.time()

    os.listdir(path)
    list_time = time.time()

    for f in tmp_files:
        os.remove(f)

    end_time = time.time()

    print(f"Create: {create_time - start:.2f} s")
    print(f"List:   {list_time - create_time:.2f} s")
    print(f"Delete: {end_time - list_time:.2f} s")


def run_suite(path, name):
    print(f"\n==============================")
    print(f"Testing {name}: {path}")
    print(f"==============================")

    for bs in BLOCK_SIZES:
        write_speeds = []
        read_speeds = []

        print(f"\nBlock size: {bs // 1024} KB")

        for i in range(REPEATS):
            print(f"  Run {i+1}...", end=" ")

            w_speed, filepath = benchmark_write(path, bs, FILE_SIZE_MB)
            r_speed = benchmark_read(filepath, bs)

            write_speeds.append(w_speed)
            read_speeds.append(r_speed)

            cleanup(filepath)
            print(f"done")

        print(f"  Write avg: {mean(write_speeds):.2f} MB/s")
        print(f"  Read avg:  {mean(read_speeds):.2f} MB/s")

    # Metadata test
    metadata_test(path)

In [4]:
run_suite(filestore_path, "Filestore")
run_suite(fuse_path, "FUSE bucket")


Testing Filestore: /home/jovyan/shared/common/JES/test

Block size: 4 KB
  Run 1... done
  Run 2... done
  Run 3... done
  Write avg: 268.87 MB/s
  Read avg:  645.91 MB/s

Block size: 64 KB
  Run 1... done
  Run 2... done
  Run 3... done
  Write avg: 300.69 MB/s
  Read avg:  672.81 MB/s

Block size: 1024 KB
  Run 1... done
  Run 2... done
  Run 3... done
  Write avg: 300.86 MB/s
  Read avg:  716.22 MB/s

Block size: 4096 KB
  Run 1... done
  Run 2... done
  Run 3... done
  Write avg: 260.16 MB/s
  Read avg:  683.25 MB/s

Metadata test (2000 files): /home/jovyan/shared/common/JES/test
Create: 10.63 s
List:   0.01 s
Delete: 4.88 s

Testing FUSE bucket: /home/jovyan/shared/disk/JES/test

Block size: 4 KB
  Run 1... done
  Run 2... done
  Run 3... done
  Write avg: 33.75 MB/s
  Read avg:  1433.06 MB/s

Block size: 64 KB
  Run 1... done
  Run 2... done
  Run 3... done
  Write avg: 130.44 MB/s
  Read avg:  3854.64 MB/s

Block size: 1024 KB
  Run 1... done
  Run 2... done
  Run 3... done
  W

## Interpretation from Copilot

These are **excellent results** — they clearly show the real differences between Filestore and GCS FUSE in your setup. Let’s interpret them properly and extract actionable conclusions.

***

# Key Takeaways

### Filestore = fast, consistent, POSIX-friendly

* Stable **260–300 MB/s writes**
* Good small-block performance
* **Metadata operations \~15× faster**

### FUSE bucket = cheaper, but high overhead

* Writes **2–8× slower** depending on block size
* Extremely poor with small I/O
* **Metadata operations \~15–20× slower**

***

# 1. Write Performance (most important)

### Filestore

* \~260–300 MB/s across all block sizes
* Very stable → good for:
  * pipelines
  * notebooks
  * shared workloads

### FUSE bucket

| Block size | Write speed             |
| ---------- | ----------------------- |
| 4 KB       | **33 MB/s (very slow)** |
| 64 KB      | 130 MB/s                |
| 1–4 MB     | \~115–120 MB/s          |

Interpretation:

* FUSE **heavily penalizes small writes**
* Even at best, it's **\~2–2.5× slower than Filestore**

***

# 2. Small I/O penalty (critical insight)

The 4 KB result:

* Filestore: **268 MB/s**
* FUSE: **33 MB/s (\~8× slower)**

This is HUGE and explains:

* slow notebooks
* laggy file saves
* inefficient Python workloads

***

# 3. Metadata Performance (this is where FUSE really breaks)

| Operation         | Filestore  | FUSE        |
| ----------------- | ---------- | ----------- |
| Create 2000 files | **10.6 s** | **167.9 s** |
| Delete 2000 files | 4.9 s      | 79.2 s      |

That’s:

* **\~16× slower create**
* **\~16× slower delete**

This is exactly what we expect:

* Filestore = real filesystem (NFS)
* FUSE = object storage pretending to be a filesystem

***

# 4. Read Speeds (still partially misleading)

You still see:

* FUSE: up to **4.7 GB/s**
* Filestore: \~650–700 MB/s

These are still **cache-inflated**, because:

* GCS FUSE does aggressive buffering
* Linux page cache is active

BUT the trend is still useful:

* Filestore → stable, moderate
* FUSE → bursty (buffer/object-based)

***

# Practical Guidance (this is the important part)

## Use Filestore for:

* Jupyter notebooks  
* Pandas / parquet datasets  
* Many small files
* Shared team workflows  
* Anything interactive

***

## Use FUSE bucket for:

* Large files (GB-scale)  
* Write-once, read-rarely data  
* Archival / cold storage  
* Batch pipelines that write in large chunks

***

## Avoid using FUSE for:

* Lots of small files  
* Frequent writes / updates  
* Python code doing many file ops  
* ML training with many samples (e.g., images)

***

# What your results imply for JupyterHub

### If users complain about:

* slow notebooks
* file save lag
* slow dataset loading

It is **very likely caused by using FUSE instead of Filestore**

***

# Real-world translation

### Example workload: pandas CSV write

| Storage   | Expected behavior      |
| --------- | ---------------------- |
| Filestore | smooth, predictable    |
| FUSE      | slower, may feel laggy |

***

### Example: many small ML files

| Storage   | Expected behavior  |
| --------- | ------------------ |
| Filestore | fine               |
| FUSE      | **painfully slow** |

***

# Best Practice Architecture (based on your results)

You now have a **textbook optimal tiered storage setup**:

### Hot storage (Filestore)

```
~/shared/common
```

* Active work
* Notebooks
* Intermediate data

### Cold storage (FUSE / GCS)

```
~/shared/disk
```

* Long-term storage
* Large results
* Final outputs

***

# Optional Optimization Tips

### 1. Improve FUSE usage

If you must use it:

* Use **large writes (≥1 MB)**
* Avoid frequent open/close cycles
* Batch operations

***

### 2. Hybrid workflow (recommended)

```text
Work locally → Filestore → sync to FUSE
```

Example:

* Train model => save to Filestore
* Move results => FUSE for cheap storage

***

### 3. Parallel workloads

Filestore will also:

* scale better under multiple users
* avoid huge contention penalties

***

# Final Verdict

Your benchmark is very solid and shows:

* Filestore = high performance, low latency  
* FUSE = cost-efficient, but not POSIX-friendly

The **16× metadata slowdown** is the most important finding  
The **8× small-I/O penalty** is the second most important

***